In [2]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import time

MARKER = 99999  

In [ ]:

def generate_task3_samples(n_samples: int, V: int, k: int, L: int, seed: int = 0):
   
    rng = np.random.default_rng(seed)

    min_len = int(np.ceil(0.9 * L))
    if min_len < 1:
        raise ValueError("L must be >= 1")
    if k < 1:
        raise ValueError("k must be >= 1")
    if min_len <= k:
        raise ValueError(f"L too small for k={k}. Need ceil(0.9*L) >= k+1.")

    sequences = []
    labels = np.empty(n_samples, dtype=np.int64)
    lengths = np.empty(n_samples, dtype=np.int64)
    marker_pos = np.empty(n_samples, dtype=np.int64)
    v_values = np.empty(n_samples, dtype=np.int64)

    for j in range(n_samples):
        l = rng.integers(min_len, L + 1)            
        m = rng.integers(0, l - k)                  

        x = rng.integers(1, V + 1, size=l, dtype=np.int64)
        x[m] = MARKER

        v = int(x[m + k])
        y = int(np.sum(x == v))

        sequences.append(x)
        labels[j] = y
        lengths[j] = l
        marker_pos[j] = m
        v_values[j] = v

    meta = {"marker_pos": marker_pos, "v_values": v_values}
    return sequences, labels, lengths, meta

In [4]:

def pad_batch(sequences, pad_value: int = 0):
 
    max_len = max(len(s) for s in sequences)
    B = len(sequences)

    x = np.full((B, max_len), pad_value, dtype=np.int64)
    mask = np.zeros((B, max_len), dtype=bool)

    for i, s in enumerate(sequences):
        x[i, :len(s)] = s
        mask[i, :len(s)] = True

    return torch.from_numpy(x), torch.from_numpy(mask)


class Task3Dataset(Dataset):
    def __init__(self, n_samples: int, V: int, k: int, L: int, seed: int = 0):
        self.seqs, self.y, self.lengths, self.meta = generate_task3_samples(
            n_samples=n_samples, V=V, k=k, L=L, seed=seed
        )

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        return torch.tensor(self.seqs[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)


def collate_task3(batch, pad_value: int = 0):
    """
    batch: list of (seq_1d, y)
    returns: x_padded [B,T], mask [B,T], y [B]
    """
    seqs, ys = zip(*batch)
    seqs_np = [s.numpy() for s in seqs]
    x_padded, mask = pad_batch(seqs_np, pad_value=pad_value)
    y = torch.stack(list(ys)).long()
    return x_padded.long(), mask, y

In [5]:

seqs, y, lengths, meta = generate_task3_samples(n_samples=3, V=10, k=2, L=20, seed=42)
for i in range(3):
    x = seqs[i]
    m = meta["marker_pos"][i]
    v = meta["v_values"][i]
    print("len=", len(x), "marker_pos=", m, "v=", v, "y=", y[i])
    print(x)
    assert x[m] == MARKER
    assert v == x[m+2]
    assert y[i] == np.sum(x == v)
print("Sanity check passed ✅")

len= 18 marker_pos= 12 v= 6 y= 2
[    7     5     5     9     1     7     3     1     6    10     8     8
 99999     8     6     2     9     5]
len= 19 marker_pos= 6 v= 5 y= 3
[    2    10     8     7     5     9 99999     5     5     3     1     6
     9     1     9     9     3     7     2]
len= 20 marker_pos= 12 v= 2 y= 2
[    4     1    10     5     9     7     8     8     2     4     5     5
 99999     6     2     8     7    10     8     4]
Sanity check passed ✅


In [6]:

def map_tokens(x_padded: torch.Tensor, V: int) -> torch.Tensor:
   
    x = x_padded.clone()
    x[x == MARKER] = V + 1
    return x

In [7]:

@torch.no_grad()
def regression_metrics(y_true: torch.Tensor, y_pred: torch.Tensor):
    y_true = y_true.float()
    y_pred = y_pred.float()
    mse = torch.mean((y_pred - y_true) ** 2).item()
    mae = torch.mean(torch.abs(y_pred - y_true)).item()
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2) + 1e-12
    r2 = (1.0 - ss_res / ss_tot).item()
    return {"MSE": mse, "MAE": mae, "R2": r2}

In [8]:

def masked_mean(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
  
    mask_f = mask.float().unsqueeze(-1)  
    x_sum = torch.sum(x * mask_f, dim=1)
    denom = torch.clamp(mask_f.sum(dim=1), min=1.0)
    return x_sum / denom

In [9]:

class RNNRegressor(nn.Module):
    def __init__(self, V: int, emb_dim: int = 64, hidden: int = 128, rnn_type: str = "lstm"):
        super().__init__()
        self.V = V
        vocab_size = V + 2  

        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        if rnn_type.lower() == "lstm":
            self.rnn = nn.LSTM(emb_dim, hidden, batch_first=True)
        elif rnn_type.lower() == "gru":
            self.rnn = nn.GRU(emb_dim, hidden, batch_first=True)
        else:
            raise ValueError("rnn_type must be 'lstm' or 'gru'")

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x_padded: torch.Tensor, mask: torch.Tensor):
        x_idx = map_tokens(x_padded, self.V)  
        e = self.emb(x_idx)                   

        lengths = mask.sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)

        
        if isinstance(h, tuple):
            h_n = h[0]
        else:
            h_n = h

        last = h_n[-1]  
        y_raw = self.head(last).squeeze(-1)   
        y_hat = F.softplus(y_raw)            
        return y_hat

In [10]:

class SelfAttentionRegressor(nn.Module):
    def __init__(self, V: int, emb_dim: int = 64, nhead: int = 4, ff_dim: int = 128, num_layers: int = 2):
        super().__init__()
        self.V = V
        vocab_size = V + 2

        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=ff_dim,
            batch_first=True,
            activation="relu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(emb_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, 1),
        )

    def forward(self, x_padded: torch.Tensor, mask: torch.Tensor):
        x_idx = map_tokens(x_padded, self.V)
        e = self.emb(x_idx)  

        key_padding = ~mask  
        h = self.encoder(e, src_key_padding_mask=key_padding)  

        pooled = masked_mean(h, mask)  
        y_raw = self.head(pooled).squeeze(-1)
        y_hat = F.softplus(y_raw)
        return y_hat

In [11]:

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_fn = nn.MSELoss()
    total_loss = 0.0
    n = 0

    for x_padded, mask, y in loader:
        x_padded = x_padded.to(device)
        mask = mask.to(device)
        y = y.float().to(device)

        optimizer.zero_grad()
        y_hat = model(x_padded, mask)
        loss = loss_fn(y_hat, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, preds = [], []
    for x_padded, mask, y in loader:
        x_padded = x_padded.to(device)
        mask = mask.to(device)
        y = y.to(device)
        y_hat = model(x_padded, mask)
        ys.append(y.float().cpu())
        preds.append(y_hat.float().cpu())

    y_true = torch.cat(ys, dim=0)
    y_pred = torch.cat(preds, dim=0)
    return regression_metrics(y_true, y_pred)


@torch.no_grad()
def measure_batch_inference_time(model, loader, device, n_batches: int = 50, warmup: int = 10):
    model.eval()
    times = []
    it = iter(loader)

    for _ in range(warmup):
        try:
            x_padded, mask, _ = next(it)
        except StopIteration:
            break
        x_padded = x_padded.to(device)
        mask = mask.to(device)
        _ = model(x_padded, mask)

   
    it = iter(loader)
    for _ in range(n_batches):
        try:
            x_padded, mask, _ = next(it)
        except StopIteration:
            break
        x_padded = x_padded.to(device)
        mask = mask.to(device)

        start = time.perf_counter()
        _ = model(x_padded, mask)
        if device.startswith("cuda"):
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append(end - start)

    return float(np.mean(times)) if times else float("nan")

In [12]:

V = 100    
k = 20     
L = 300     

n_train = 20000
n_test  = 5000
batch_size = 128

train_ds = Task3Dataset(n_samples=n_train, V=V, k=k, L=L, seed=0)
test_ds  = Task3Dataset(n_samples=n_test,  V=V, k=k, L=L, seed=1)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_task3)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_task3)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [13]:

model_lstm = RNNRegressor(V=V, emb_dim=64, hidden=128, rnn_type="lstm").to(device)
opt = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)

epochs = 15
for ep in range(1, epochs + 1):
    tr_mse = train_one_epoch(model_lstm, train_loader, opt, device)
    te = evaluate(model_lstm, test_loader, device)
    t_inf = measure_batch_inference_time(model_lstm, test_loader, device, n_batches=50)
    print(f"[LSTM] ep {ep:02d} | train MSE {tr_mse:.4f} | test MAE {te['MAE']:.4f} MSE {te['MSE']:.4f} R2 {te['R2']:.4f} | batch_time {t_inf*1000:.2f} ms")

[LSTM] ep 01 | train MSE 3.7995 | test MAE 1.3462 MSE 2.8635 R2 -0.0023 | batch_time 71.06 ms
[LSTM] ep 02 | train MSE 2.8530 | test MAE 1.3610 MSE 2.9082 R2 -0.0180 | batch_time 69.43 ms
[LSTM] ep 03 | train MSE 2.8598 | test MAE 1.3568 MSE 2.8819 R2 -0.0088 | batch_time 72.27 ms
[LSTM] ep 04 | train MSE 2.8570 | test MAE 1.3508 MSE 2.8677 R2 -0.0038 | batch_time 72.58 ms
[LSTM] ep 05 | train MSE 2.8516 | test MAE 1.3788 MSE 2.9422 R2 -0.0299 | batch_time 69.44 ms
[LSTM] ep 06 | train MSE 2.8480 | test MAE 1.3503 MSE 2.8828 R2 -0.0091 | batch_time 69.01 ms
[LSTM] ep 07 | train MSE 2.8398 | test MAE 1.3582 MSE 2.8964 R2 -0.0139 | batch_time 68.54 ms
[LSTM] ep 08 | train MSE 2.8337 | test MAE 1.3481 MSE 2.8685 R2 -0.0041 | batch_time 69.06 ms
[LSTM] ep 09 | train MSE 2.8273 | test MAE 1.3530 MSE 2.8763 R2 -0.0068 | batch_time 69.99 ms
[LSTM] ep 10 | train MSE 2.8288 | test MAE 1.3563 MSE 2.8905 R2 -0.0118 | batch_time 70.01 ms
[LSTM] ep 11 | train MSE 2.8341 | test MAE 1.3589 MSE 2.9076

In [14]:
# CELL 13 — Train GRU (EN/FA)
model_gru = RNNRegressor(V=V, emb_dim=64, hidden=128, rnn_type="gru").to(device)
opt = torch.optim.Adam(model_gru.parameters(), lr=1e-3)

epochs = 15
for ep in range(1, epochs + 1):
    tr_mse = train_one_epoch(model_gru, train_loader, opt, device)
    te = evaluate(model_gru, test_loader, device)
    t_inf = measure_batch_inference_time(model_gru, test_loader, device, n_batches=50)
    print(f"[GRU ] ep {ep:02d} | train MSE {tr_mse:.4f} | test MAE {te['MAE']:.4f} MSE {te['MSE']:.4f} R2 {te['R2']:.4f} | batch_time {t_inf*1000:.2f} ms")

[GRU ] ep 01 | train MSE 3.7037 | test MAE 1.3489 MSE 2.8596 R2 -0.0010 | batch_time 61.65 ms
[GRU ] ep 02 | train MSE 2.8767 | test MAE 1.3453 MSE 2.8631 R2 -0.0022 | batch_time 218.88 ms
[GRU ] ep 03 | train MSE 2.8465 | test MAE 1.3456 MSE 2.8660 R2 -0.0032 | batch_time 59.25 ms
[GRU ] ep 04 | train MSE 2.8371 | test MAE 1.3532 MSE 2.8747 R2 -0.0062 | batch_time 59.20 ms
[GRU ] ep 05 | train MSE 2.8303 | test MAE 1.3494 MSE 2.8689 R2 -0.0042 | batch_time 59.41 ms
[GRU ] ep 06 | train MSE 2.8327 | test MAE 1.3509 MSE 2.8751 R2 -0.0064 | batch_time 59.36 ms
[GRU ] ep 07 | train MSE 2.8091 | test MAE 1.3569 MSE 2.8898 R2 -0.0115 | batch_time 59.49 ms
[GRU ] ep 08 | train MSE 2.8000 | test MAE 1.3655 MSE 2.9762 R2 -0.0418 | batch_time 58.16 ms
[GRU ] ep 09 | train MSE 2.7691 | test MAE 1.3653 MSE 2.9237 R2 -0.0234 | batch_time 57.78 ms
[GRU ] ep 10 | train MSE 2.7266 | test MAE 1.3675 MSE 2.9453 R2 -0.0310 | batch_time 58.10 ms
[GRU ] ep 11 | train MSE 2.6638 | test MAE 1.3743 MSE 3.032

In [15]:
# CELL 14 — Train Attention (TransformerEncoder) (EN/FA)
model_attn = SelfAttentionRegressor(V=V, emb_dim=64, nhead=4, ff_dim=128, num_layers=2).to(device)
opt = torch.optim.Adam(model_attn.parameters(), lr=1e-3)

epochs = 15
for ep in range(1, epochs + 1):
    tr_mse = train_one_epoch(model_attn, train_loader, opt, device)
    te = evaluate(model_attn, test_loader, device)
    t_inf = measure_batch_inference_time(model_attn, test_loader, device, n_batches=50)
    print(f"[ATTN] ep {ep:02d} | train MSE {tr_mse:.4f} | test MAE {te['MAE']:.4f} MSE {te['MSE']:.4f} R2 {te['R2']:.4f} | batch_time {t_inf*1000:.2f} ms")

c:\Users\Dorsa Abbasi\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


[ATTN] ep 01 | train MSE 3.2856 | test MAE 1.3403 MSE 2.8742 R2 -0.0061 | batch_time 178.34 ms
[ATTN] ep 02 | train MSE 2.8513 | test MAE 1.3500 MSE 2.8570 R2 -0.0001 | batch_time 179.07 ms
[ATTN] ep 03 | train MSE 2.8506 | test MAE 1.3718 MSE 2.9254 R2 -0.0240 | batch_time 181.94 ms
[ATTN] ep 04 | train MSE 2.8475 | test MAE 1.3480 MSE 2.8573 R2 -0.0001 | batch_time 178.72 ms
[ATTN] ep 05 | train MSE 2.8491 | test MAE 1.3734 MSE 2.9278 R2 -0.0248 | batch_time 179.70 ms
[ATTN] ep 06 | train MSE 2.8555 | test MAE 1.3434 MSE 2.8644 R2 -0.0026 | batch_time 178.81 ms
[ATTN] ep 07 | train MSE 2.8485 | test MAE 1.3539 MSE 2.8616 R2 -0.0017 | batch_time 179.05 ms
[ATTN] ep 08 | train MSE 2.8503 | test MAE 1.3414 MSE 2.8701 R2 -0.0047 | batch_time 178.14 ms
[ATTN] ep 09 | train MSE 2.8441 | test MAE 1.3444 MSE 2.8622 R2 -0.0019 | batch_time 178.75 ms
[ATTN] ep 10 | train MSE 2.8599 | test MAE 1.3450 MSE 2.8607 R2 -0.0013 | batch_time 178.19 ms
[ATTN] ep 11 | train MSE 2.8497 | test MAE 1.3498 

In [16]:
# CELL 15 — Try a few predictions (EN/FA)
@torch.no_grad()
def predict_some(model, dataset, n=5):
    model.eval()
    for i in range(n):
        x, y = dataset[i]
        x_padded, mask = pad_batch([x.numpy()], pad_value=0)
        x_padded = x_padded.to(device)
        mask = mask.to(device)
        y_hat = model(x_padded, mask).cpu().item()
        print(f"true y={int(y.item())}, pred={y_hat:.2f}, len={len(x)}")

print("LSTM predictions:")
predict_some(model_lstm, test_ds, n=8)

LSTM predictions:
true y=4, pred=3.69, len=284
true y=2, pred=3.78, len=278
true y=3, pred=4.13, len=286
true y=3, pred=3.61, len=292
true y=3, pred=4.11, len=281
true y=2, pred=3.47, len=282
true y=4, pred=3.59, len=270
true y=4, pred=4.13, len=276
